# Category Embeddings (Entity Embeddings) in Keras

## Introduction
In this notebook, we explore a powerful technique for handling categorical variables in deep learning: **Entity Embeddings**.

Traditionally, categorical variables are handled using One-Hot Encoding. However, for features with high cardinality (many unique categories, such as User IDs, Product IDs, or Zip Codes), One-Hot Encoding leads to massive, sparse vectors that are memory-inefficient and struggle to capture relationships between categories.

**Embeddings** solve this by learning a low-dimensional, dense vector representation for each category. These vectors are learned during training, meaning the model places similar categories closer together in the embedding space.

### Objectives
- Understand the limitations of One-Hot Encoding.
- Master the `tf.keras.layers.Embedding` layer.
- Implement a model using Entity Embeddings on a dataset.
- Visualize the learned embeddings.

---

## 1. One-Hot vs. Embeddings

### The Problem: High Cardinality
Imagine representing "Team ID" in a league with 350 teams. 
- **One-Hot**: A vector of size 350 with one '1' and 349 '0's. Sparse and large.
- **Embedding**: A vector of size (e.g.) 10 with dense float numbers. Compact and rich in information.

### Visual Comparison
The diagram below illustrates the concept (often applied to words, but the same logic applies to any category):

![Embedding Lookup](https://miro.medium.com/v2/resize:fit:1400/1*yA5l7d_z0E04kF8Q6kYJzQ.png)
*(Note: Similar concept to the MLP diagram, inputs are transformed into dense layers)*

A better visualization of the Lookup Table concept:
![Embedding Lookup Table](https://upload.wikimedia.org/wikipedia/commons/thumb/e/e4/Artificial_neural_network.svg/640px-Artificial_neural_network.svg.png)
*(Standard ANN structure, where the first weight matrix acts as the embedding lookup)*

### Concept: The Lookup Table
Technically, an Embedding layer is just a learnable lookup table. 
- **Input**: Integer index of the category (e.g., Team ID `5`)
- **Operation**: Look up row `5` in the matrix.
- **Output**: The vector at row `5`.


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# Set seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

---
## 2. Loading the Dataset
We will use the **Basketball Data** to predict game outcomes based on the teams playing.
Our categorical features are the **Team IDs**.


In [ ]:
# Load the dataset
df = pd.read_csv('../../00-data/games_season.csv')
print("Shape:", df.shape)
df.head()

### Preprocessing
We need to feed integer IDs to the Embedding layer. The 'team_1' and 'team_2' columns likely contain team abbreviations or loose IDs. It's safer to map them to a strict range `[0, N-1]`.

In [ ]:
# Get inputs and target
input_1 = df['team_1']
input_2 = df['team_2']
target = df['score_diff'] # We will predict the score difference

# Determine unique teams to find vocabulary size (N)
# We combine both columns to ensure we cover all teams
unique_teams = np.unique(np.concatenate([input_1.unique(), input_2.unique()]))
num_unique_teams = len(unique_teams)

print(f"Number of unique teams: {num_unique_teams}")

# Create a mapping from Team ID to Integer Index [0, N-1]
team_to_int = {team: i for i, team in enumerate(unique_teams)}

# Apply mapping
input_1_int = input_1.map(team_to_int)
input_2_int = input_2.map(team_to_int)

---
## 3. The Embedding Layer

### Keras Documentation
`tf.keras.layers.Embedding(input_dim, output_dim, input_length)`

- **`input_dim`**: Size of the vocabulary (number of unique categories). Here, `num_unique_teams`.
- **`output_dim`**: Dimension of the dense embedding vector. A hyperparameter we choose (e.g., 8, 50).
- **`input_length`**: Length of input sequences (1 for us, as we feed one team at a time).


In [ ]:
# Define Embedding Size
# Rule of thumb: min(50, num_categories/2) or 4th root of N. Let's pick 8 for simplicity.
embed_dim = 8

print(f"Embedding Input Dim: {num_unique_teams}")
print(f"Embedding Output Dim: {embed_dim}")

### How to Choose Embedding Dimension

Choosing the right embedding dimension is important:

**Rules of Thumb:**
- **Minimum**: `4th root of num_categories` (e.g., √√350 ≈ 4.3)
- **Maximum**: `min(50, num_categories // 2)`
- **Common practice**: Start with 8-32 for most use cases

**Trade-offs:**
| Dimension | Pros | Cons |
|-----------|------|------|
| **Too Small** (e.g., 2) | Fast training, less overfitting | Can't capture complex relationships |
| **Too Large** (e.g., 100) | Rich representations | Slow, overfitting risk, memory usage |
| **Just Right** (e.g., 8-32) | Good balance | Requires experimentation |

**For our ~350 teams:**
- 4th root: √√350 ≈ 4.3
- We use **8** as a reasonable starting point
- Experiment with 4, 8, 16, 32 and compare validation loss

---
## 4. Building the Model
We will use the **Functional API** because we have two distinct inputs (Team 1 and Team 2) that share the *same* embedding layer ( since they are from the same pool of teams).

In [ ]:
# 1. Define Inputs
team_in_1 = layers.Input(shape=(1,), name='Team1_Input')
team_in_2 = layers.Input(shape=(1,), name='Team2_Input')

# 2. Define Shared Embedding Layer
team_embedding_layer = layers.Embedding(input_dim=num_unique_teams, 
                                        output_dim=embed_dim, 
                                        input_length=1, 
                                        name='Team_Embedding')

# 3. Lookup Embeddings for both inputs
# Output shape: (Batch, 1, embed_dim)
team_1_embed = team_embedding_layer(team_in_1)
team_2_embed = team_embedding_layer(team_in_2)

# 4. Flatten to remove the extra sequence dimension
# Output shape: (Batch, embed_dim)
team_1_flat = layers.Flatten()(team_1_embed)
team_2_flat = layers.Flatten()(team_2_embed)

# 5. Combine and Predict
# We can concatenate the vectors
concat = layers.Concatenate()([team_1_flat, team_2_flat])

# Dense layers for regression
x = layers.Dense(64, activation='relu')(concat)
out = layers.Dense(1, name='Score_Diff_Pred')(x)

# Create Model
model = models.Model(inputs=[team_in_1, team_in_2], outputs=out)
model.summary()

### Visualizing the Architecture
Notice how the 'Team_Embedding' layer is connected to both inputs.

In [ ]:
try:
    tf.keras.utils.plot_model(model, show_shapes=True)
except Exception as e:
    print('plot_model needs graphviz:', e)

In [ ]:
# Compile and Train
model.compile(optimizer='adam', loss='mae') # Mean Absolute Error for regression

history = model.fit(
    [input_1_int, input_2_int],
    target,
    epochs=3,
    batch_size=2048,
    validation_split=0.1
)

---
## 5. Exercises

### Exercise 1: Change Embedding Dimension
Re-build the model but increase the `output_dim` of the embedding layer to 50. Does the model train slower or faster? (Note the increase in parameters).


In [ ]:
# Exercise 1 Solution: Change Embedding Dimension to 50
embed_dim_50 = 50

# Rebuild the model with larger embeddings
team_in_1_ex = layers.Input(shape=(1,), name='Team1_Input_Ex1')
team_in_2_ex = layers.Input(shape=(1,), name='Team2_Input_Ex1')

team_embedding_50 = layers.Embedding(input_dim=num_unique_teams, 
                                      output_dim=embed_dim_50, 
                                      input_length=1, 
                                      name='Team_Embedding_50')

team_1_embed_50 = team_embedding_50(team_in_1_ex)
team_2_embed_50 = team_embedding_50(team_in_2_ex)

team_1_flat_50 = layers.Flatten()(team_1_embed_50)
team_2_flat_50 = layers.Flatten()(team_2_embed_50)

concat_50 = layers.Concatenate()([team_1_flat_50, team_2_flat_50])
x_50 = layers.Dense(64, activation='relu')(concat_50)
out_50 = layers.Dense(1, name='Score_Diff_Pred_50')(x_50)

model_50 = models.Model(inputs=[team_in_1_ex, team_in_2_ex], outputs=out_50)
model_50.compile(optimizer='adam', loss='mae')

print("Model with embed_dim=50:")
print(f"Total parameters: {model_50.count_params()}")
# Note: More parameters = slower training, but potentially better representation

### Exercise 2: Tourney Prediction
Load `Basketball Data/games_tourney.csv`. Use the trained model to predict score differences for tournament games.
1. Load the csv.
2. Preprocess team columns using the **same** `team_to_int` mapping created earlier. (Note: Handle potential unknown teams if any).
3. Use `model.predict()`.

In [ ]:
# Exercise 2 Solution: Tourney Prediction
df_tourney = pd.read_csv('../../00-data/games_tourney.csv')
print("Tourney data shape:", df_tourney.shape)

# Preprocess using the SAME mapping
tourney_team_1 = df_tourney['team_1'].map(team_to_int)
tourney_team_2 = df_tourney['team_2'].map(team_to_int)

# Handle any unknown teams (teams in tourney but not in season data)
unknown_mask = tourney_team_1.isna() | tourney_team_2.isna()
if unknown_mask.any():
    print(f"Warning: {unknown_mask.sum()} games have unknown teams, skipping...")
    tourney_team_1 = tourney_team_1[~unknown_mask]
    tourney_team_2 = tourney_team_2[~unknown_mask]

# Predict using the trained model
predictions = model.predict([tourney_team_1, tourney_team_2])
print(f"Sample predictions: {predictions[:5].flatten()}")